# 6. Prediction and Deployment

How the trained models are used: `predict_trust` scores one user with the right model for their role, and the Flask service in `app.py` exposes that over HTTP for the backend.

## Setup

Runs locally or in Google Colab. In Colab it mounts Google Drive, copies this repo's code from GitHub, and points `data/` at **My Drive** (raw files in the Drive root, outputs in `My Drive/processed`) and `models/` at `My Drive/models`, so outputs survive between sessions.

In [1]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_URL = "https://github.com/JamieNguru/169421-escrow-based-wallet-and-payment-system.git"
    if not Path("/content/repo").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "dev", REPO_URL, "/content/repo"], check=True)
    ROOT = Path("/content/repo/trust-scoring-ml")
    DRIVE = Path("/content/drive/MyDrive")
    (DRIVE / "models").mkdir(exist_ok=True)
    for name, target in [("data", DRIVE), ("models", DRIVE / "models")]:
        if not (ROOT / name).exists():
            (ROOT / name).symlink_to(target, target_is_directory=True)
else:
    ROOT = Path.cwd().parent

sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd

from src.predict import FEATURE_COLUMNS_BY_ROLE, predict_trust

## Required features per role

In [3]:
for role, columns in FEATURE_COLUMNS_BY_ROLE.items():
    print(f"{role}: {columns}")

worker: ['job_completion_rate', 'dispute_rate', 'response_time_hours']
client: ['payment_completion_rate', 'refund_rate', 'escrow_release_time_hours']


## Scoring example users

In [4]:
examples = [
    ("reliable worker", "worker", {"job_completion_rate": 0.995, "dispute_rate": 0.005, "response_time_hours": 5.0}),
    ("unreliable worker", "worker", {"job_completion_rate": 0.950, "dispute_rate": 0.050, "response_time_hours": 30.0}),
    ("prompt client", "client", {"payment_completion_rate": 0.995, "refund_rate": 0.005, "escrow_release_time_hours": 5.0}),
    ("slow client", "client", {"payment_completion_rate": 0.950, "refund_rate": 0.050, "escrow_release_time_hours": 30.0}),
]

rows = []
for name, role, features in examples:
    result = predict_trust(role, features)
    rows.append({"user": name, "role": role, "trust_level": result["trust_level"],
                 **{f"P({k})": round(float(v), 3) for k, v in result["probabilities"].items()}})
pd.DataFrame(rows).set_index("user")

,role,trust_level,P(High),P(Low),P(Medium)
user,,,,,
reliable worker,worker,High,1.0,0.0,0.0
unreliable worker,worker,Low,0.0,1.0,0.0
prompt client,client,High,1.0,0.0,0.0
slow client,client,Low,0.0,1.0,0.0


## Invalid requests are rejected

In [5]:
for role, features in [
    ("admin", {}),
    ("worker", {"job_completion_rate": 0.99}),
]:
    try:
        predict_trust(role, features)
    except ValueError as error:
        print(f"{role}: {error}")

admin: Unknown role 'admin'; expected one of ['worker', 'client']
worker: Missing required features for role 'worker': ['dispute_rate', 'response_time_hours']


## The Flask service

`app.py` serves `GET /health` and `POST /score`. Here it's called through Flask's test client; run `python app.py` to serve it for real at `http://127.0.0.1:5000`, which is what the backend calls when it needs a trust score (issue #21).

In [6]:
from app import app

client = app.test_client()

response = client.get("/health")
print("GET /health ->", response.status_code, response.get_json())

for body in [
    {"role": "worker", "features": examples[0][2]},
    {"role": "client", "features": examples[3][2]},
    {"role": "worker", "features": {"job_completion_rate": 0.99}},
]:
    response = client.post("/score", json=body)
    print("\nPOST /score", body)
    print("  ->", response.status_code, response.get_json())

GET /health -> 200 {'status': 'ok'}

POST /score {'role': 'worker', 'features': {'job_completion_rate': 0.995, 'dispute_rate': 0.005, 'response_time_hours': 5.0}}
  -> 200 {'probabilities': {'High': 1.0, 'Low': 0.0, 'Medium': 0.0}, 'role': 'worker', 'trust_level': 'High'}



POST /score {'role': 'client', 'features': {'payment_completion_rate': 0.95, 'refund_rate': 0.05, 'escrow_release_time_hours': 30.0}}
  -> 200 {'probabilities': {'High': 0.0, 'Low': 1.0, 'Medium': 0.0}, 'role': 'client', 'trust_level': 'Low'}

POST /score {'role': 'worker', 'features': {'job_completion_rate': 0.99}}
  -> 400 {'error': "Missing required features for role 'worker': ['dispute_rate', 'response_time_hours']"}
